# LogRMS 5M TinyStories compiled parity run

> **Build:** `LOGRMS-5M-COMPILED-PARITY-20260714`

This self-contained Colab trains the compiled LogRMS optimizer for two complete passes over the official TinyStories dataset. It embeds the current reference and compiled sources, verifies their hashes, and compares the run with reference telemetry at matching steps and token counts.


In [ ]:
import sys, subprocess

REQUIRED = [
    "datasets>=2.20,<4",
    "sentencepiece>=0.2,<1",
    "torchao==0.17.0",
    "psutil>=5.9",
    "pandas>=2.0",
    "matplotlib>=3.7",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])


In [ ]:
from __future__ import annotations

import base64
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterator

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset, Sampler


@dataclass(reference=True)
class Config:
    dataset_id: str = "roneneldan/TinyStories"
    vocab_size: int = 4096
    sequence_length: int = 256
    d_model: int = 256
    heads: int = 8
    layers: int = 5
    feedforward_dim: int = 1024
    dropout: float = 0.0
    batch_size: int = 32
    epochs: int = 2
    tune_tokens: int = 20_000_000
    tune_seed: int = 1701
    final_seed: int = 2701
    data_seed: int = 3701
    weight_decay: float = 0.01
    beta2: float = 0.999
    eps: float = 1e-8
    max_grad_norm: float = 1.0
    warmup_fraction: float = 0.05
    min_lr_ratio: float = 0.01
    log_every: int = 50
    eval_every: int = 1000
    checkpoint_every: int = 2000
    eval_batches: int = 128
    num_workers: int = 2
    persist_cache_in_drive: bool = True
    force_rebuild_data: bool = False
    run_tuning: bool = False
    run_final: bool = True


CFG = Config()
NOTEBOOK_BUILD = "LOGRMS-5M-COMPILED-PARITY-20260714"
REFERENCE_OPTIMIZER = "logrms_reference"
COMPILED_OPTIMIZER = "logrms_compiled"
PEAK_LR = 3e-3
assert CFG.epochs == 2, "The registered protocol is exactly two epochs."
print("NOTEBOOK BUILD:", NOTEBOOK_BUILD)


## Storage and preflight

Google Drive is used for checkpoints and results when mounted. Set
`persist_cache_in_drive=False` above if Drive space is tight; the token cache
then lives only in the Colab VM.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/logrms_tinystories_5m")
except Exception:
    DRIVE_ROOT = Path("/content/logrms_tinystories_5m")

LOCAL_ROOT = Path("/content/logrms_tinystories_5m_local")
RUN_ROOT = DRIVE_ROOT / "runs"
DATA_ROOT = (DRIVE_ROOT if CFG.persist_cache_in_drive else LOCAL_ROOT) / "data"
for path in (DRIVE_ROOT, LOCAL_ROOT, RUN_ROOT, DATA_ROOT):
    path.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab: Runtime > Change runtime type > GPU")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
print({
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "amp_dtype": str(AMP_DTYPE),
    "data_root": str(DATA_ROOT),
    "run_root": str(RUN_ROOT),
})


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path: Path, obj: Any) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(temp, path)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


MANIFEST = {
    "config": asdict(CFG),
    "registered_peak_lr": PEAK_LR,
    "reference_optimizer": REFERENCE_OPTIMIZER,
    "compiled_optimizer": COMPILED_OPTIMIZER,
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "torchao": importlib.metadata.version("torchao"),
    "datasets": importlib.metadata.version("datasets"),
    "sentencepiece": importlib.metadata.version("sentencepiece"),
    "precision": str(AMP_DTYPE),
}
if MANIFEST["torchao"] != "0.17.0":
    raise RuntimeError(f"Protocol requires torchao 0.17.0, found {MANIFEST['torchao']}")


## Download, tokenize, and pack the complete official corpus

SentencePiece is trained once on a deterministic prefix of the training split.
The complete train split is then encoded into a contiguous `uint16` stream,
with EOS between stories. The official validation split is divided by story
index into fixed validation and held-out test halves. Packing is lossless: the
last short block is padded and excluded from the loss rather than discarded.


In [ ]:
TOKENIZER_PREFIX = DATA_ROOT / "tinystories_bpe4096"
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix(".model")
TOKENIZER_TEXT = DATA_ROOT / "tokenizer_training_text.txt"
TOKEN_LIMIT_CHARS = 50_000_000


def get_splits():
    train = load_dataset(CFG.dataset_id, split="train")
    official_validation = load_dataset(CFG.dataset_id, split="validation")
    midpoint = len(official_validation) // 2
    validation = official_validation.select(range(0, midpoint))
    test = official_validation.select(range(midpoint, len(official_validation)))
    return train, validation, test


train_split, validation_split, test_split = get_splits()
print({"train_stories": len(train_split), "validation_stories": len(validation_split), "test_stories": len(test_split)})


def build_tokenizer() -> spm.SentencePieceProcessor:
    if CFG.force_rebuild_data or not TOKENIZER_MODEL.exists():
        written = 0
        with TOKENIZER_TEXT.open("w", encoding="utf-8") as out:
            for row in train_split:
                text = row["text"]
                remaining = TOKEN_LIMIT_CHARS - written
                if remaining <= 0:
                    break
                text = text[:remaining]
                out.write(text.replace("\x00", " ") + "\n")
                written += len(text)
        spm.SentencePieceTrainer.train(
            input=str(TOKENIZER_TEXT),
            model_prefix=str(TOKENIZER_PREFIX),
            vocab_size=CFG.vocab_size,
            model_type="bpe",
            character_coverage=1.0,
            pad_id=0,
            unk_id=1,
            eos_id=2,
            bos_id=-1,
            byte_fallback=True,
            num_threads=max(1, os.cpu_count() or 1),
            shuffle_input_sentence=False,
            input_sentence_size=0,
        )
    processor = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
    if processor.vocab_size() != CFG.vocab_size:
        raise RuntimeError("Tokenizer vocabulary does not match protocol")
    return processor


SP = build_tokenizer()
MANIFEST["tokenizer_sha256"] = sha256_file(TOKENIZER_MODEL)


def encode_split(dataset, name: str) -> dict[str, Any]:
    binary = DATA_ROOT / f"{name}.uint16.bin"
    metadata = DATA_ROOT / f"{name}.json"
    if binary.exists() and metadata.exists() and not CFG.force_rebuild_data:
        info = json.loads(metadata.read_text())
        if info.get("sha256") == sha256_file(binary):
            return info

    temp = binary.with_suffix(binary.suffix + ".tmp")
    token_count = 0
    with temp.open("wb") as out:
        for start in range(0, len(dataset), 1024):
            texts = dataset[start : start + 1024]["text"]
            encoded = SP.encode(texts, out_type=int)
            for ids in encoded:
                values = np.asarray([*ids, SP.eos_id()], dtype=np.uint16)
                values.tofile(out)
                token_count += values.size
            if start and start % 100_000 == 0:
                print(name, f"{start:,}/{len(dataset):,} stories", f"{token_count:,} tokens")
    os.replace(temp, binary)
    info = {
        "name": name,
        "stories": len(dataset),
        "tokens": int(token_count),
        "bytes": binary.stat().st_size,
        "sha256": sha256_file(binary),
        "path": str(binary),
    }
    atomic_json(metadata, info)
    return info


SPLIT_INFO = {
    "train": encode_split(train_split, "train"),
    "validation": encode_split(validation_split, "validation"),
    "test": encode_split(test_split, "test"),
}
MANIFEST["splits"] = SPLIT_INFO
print(pd.DataFrame(SPLIT_INFO).T)


In [ ]:
class PackedTokenDataset(Dataset):
    def __init__(self, info: dict[str, Any], sequence_length: int):
        self.tokens = np.memmap(info["path"], mode="r", dtype=np.uint16)
        self.sequence_length = sequence_length
        self.target_count = max(0, len(self.tokens) - 1)
        self.block_count = math.ceil(self.target_count / sequence_length)

    def __len__(self) -> int:
        return self.block_count

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        start = index * self.sequence_length
        stop = min(start + self.sequence_length + 1, len(self.tokens))
        raw = np.asarray(self.tokens[start:stop], dtype=np.int64)
        real_targets = max(0, raw.size - 1)
        values = np.full(self.sequence_length + 1, 0, dtype=np.int64)
        values[: raw.size] = raw
        inputs = torch.from_numpy(values[:-1].copy())
        labels = torch.from_numpy(values[1:].copy())
        labels[real_targets:] = -100
        return inputs, labels, real_targets


class FixedEpochSampler(Sampler[int]):
    def __init__(self, size: int, seed: int, epoch: int, start_index: int = 0):
        generator = torch.Generator().manual_seed(seed + epoch)
        self.order = torch.randperm(size, generator=generator).tolist()[start_index:]

    def __iter__(self) -> Iterator[int]:
        return iter(self.order)

    def __len__(self) -> int:
        return len(self.order)


TRAIN_DATA = PackedTokenDataset(SPLIT_INFO["train"], CFG.sequence_length)
VALID_DATA = PackedTokenDataset(SPLIT_INFO["validation"], CFG.sequence_length)
TEST_DATA = PackedTokenDataset(SPLIT_INFO["test"], CFG.sequence_length)
print({"train_blocks": len(TRAIN_DATA), "targets_per_epoch": TRAIN_DATA.target_count})


## Model

This is a tied-embedding, pre-norm causal transformer using PyTorch scaled-dot-
product attention. Parameter tensors remain FP32; activations use BF16 where
supported, otherwise FP16 with gradient scaling.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, heads: int, dropout: float):
        super().__init__()
        if d_model % heads:
            raise ValueError("d_model must be divisible by heads")
        self.heads = heads
        self.head_dim = d_model // heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, width = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        return self.proj(y.transpose(1, 2).contiguous().view(batch, length, width))


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(CFG.d_model)
        self.attn = CausalSelfAttention(CFG.d_model, CFG.heads, CFG.dropout)
        self.ln2 = nn.LayerNorm(CFG.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(CFG.d_model, CFG.feedforward_dim),
            nn.GELU(),
            nn.Linear(CFG.feedforward_dim, CFG.d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class TinyStoriesLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(CFG.vocab_size, CFG.d_model)
        self.position_embedding = nn.Embedding(CFG.sequence_length, CFG.d_model)
        self.blocks = nn.ModuleList([Block() for _ in range(CFG.layers)])
        self.final_norm = nn.LayerNorm(CFG.d_model)
        self.output = nn.Linear(CFG.d_model, CFG.vocab_size, bias=False)
        self.apply(self._init)
        self.output.weight = self.token_embedding.weight

    @staticmethod
    def _init(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, 0.0, 0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(tokens.shape[1], device=tokens.device)
        x = self.token_embedding(tokens) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        return self.output(self.final_norm(x))


def make_initial_state(seed: int) -> tuple[dict[str, torch.Tensor], str, int]:
    seed_everything(seed)
    template = TinyStoriesLM()
    count = sum(p.numel() for p in template.parameters())
    state = {k: v.detach().cpu().clone() for k, v in template.state_dict().items()}
    digest = hashlib.sha256(b"".join(v.numpy().tobytes() for v in state.values())).hexdigest()
    return state, digest, count


TUNE_INITIAL_STATE, TUNE_INITIAL_HASH, PARAMETER_COUNT = make_initial_state(CFG.tune_seed)
FINAL_INITIAL_STATE, FINAL_INITIAL_HASH, final_count = make_initial_state(CFG.final_seed)
if final_count != PARAMETER_COUNT or not 4_000_000 <= PARAMETER_COUNT <= 6_500_000:
    raise RuntimeError(f"Expected an approximately 5M-parameter model, found {PARAMETER_COUNT:,}")
print({
    "parameters": f"{PARAMETER_COUNT:,}",
    "tuning_initial_state_sha256": TUNE_INITIAL_HASH,
    "final_initial_state_sha256": FINAL_INITIAL_HASH,
})


In [ ]:
# Current LogRMS reference source, embedded for a self-contained run.
REFERENCE_SOURCE_B64 = '...'


In [ ]:
REFERENCE_PATH = Path("/content/logrms_reference.py")
REFERENCE_SHA256 = "1a59b6b948d565c646a435ad9c97f84b95579c347b981a32ebd81b119858e4f4"
REFERENCE_PATH.write_bytes(base64.b64decode(REFERENCE_SOURCE_B64))
if sha256_file(REFERENCE_PATH) != REFERENCE_SHA256:
    raise RuntimeError("Embedded LogRMS reference source hash mismatch")
import importlib.util
reference_spec = importlib.util.spec_from_file_location("logrms_reference", REFERENCE_PATH)
logrms_reference = importlib.util.module_from_spec(reference_spec)
assert reference_spec.loader is not None
sys.modules[reference_spec.name] = logrms_reference
reference_spec.loader.exec_module(logrms_reference)
LogRMS = logrms_reference.LogRMS
CONFIDENCE_DEFAULT = logrms_reference.CONFIDENCE_DEFAULT
print("LogRMS reference", logrms_reference.VERSION, sha256_file(REFERENCE_PATH))


In [ ]:
# Current LogRMS compiled source, embedded for a self-contained run.
COMPILED_SOURCE_B64 = '...'


In [ ]:
COMPILED_PATH = Path("/content/logrms_compiled.py")
COMPILED_PATH.write_bytes(base64.b64decode(COMPILED_SOURCE_B64))
compiled_spec = importlib.util.spec_from_file_location("logrms_compiled", COMPILED_PATH)
logrms_compiled = importlib.util.module_from_spec(compiled_spec)
assert compiled_spec.loader is not None
sys.modules[compiled_spec.name] = logrms_compiled
compiled_spec.loader.exec_module(logrms_compiled)
LogRMSCompiled = logrms_compiled.LogRMSCompiled
print({
    "compiled_version": logrms_compiled.COMPILED_VERSION,
    "reference_version": logrms_compiled.REFERENCE_VERSION,
    "compiled_source_sha256": sha256_file(COMPILED_PATH),
})


## Optimizers, schedule, auditing, and evaluation


In [ ]:
from torchao.optim import AdamW8bit
from torchao.optim.subclass_8bit import OptimState8bit


def parameter_groups(model: nn.Module) -> list[dict[str, Any]]:
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return [
        {"params": decay, "weight_decay": CFG.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def make_optimizer(name: str, model: nn.Module, peak_lr: float):
    if name != COMPILED_OPTIMIZER:
        raise KeyError(name)
    return LogRMSCompiled(
        parameter_groups(model),
        lr=peak_lr,
        beta2=CFG.beta2,
        eps=CFG.eps,
        backend="8bit",
        confidence_table=CONFIDENCE_DEFAULT,
        max_norm=CFG.max_grad_norm,
        stochastic_rounding=True,
        rounding_seed=CFG.final_seed,
        compile_backend="inductor",
        compile_mode="reduce-overhead",
        fullgraph=True,
        dynamic=False,
        fallback_on_compile_error=False,
    )


def scheduled_lr(peak_lr: float, step: int, total_steps: int) -> float:
    warmup = max(1, round(total_steps * CFG.warmup_fraction))
    if step < warmup:
        return peak_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, total_steps - warmup - 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return peak_lr * (CFG.min_lr_ratio + (1.0 - CFG.min_lr_ratio) * cosine)


def set_optimizer_lr(optimizer, lr: float) -> None:
    """Apply the registered schedule without changing optimizer semantics."""
    for group in optimizer.param_groups:
        if torch.is_tensor(group["lr"]):
            group["lr"].fill_(lr)
        else:
            group["lr"] = lr


def optimizer_state_audit(optimizer) -> tuple[int, float]:
    """Count unique physical tensor storage, including TorchAO tensor subclasses."""
    seen, total, quantized = set(), 0, 0

    def add(tensor: torch.Tensor) -> None:
        nonlocal total
        storage = tensor.untyped_storage()
        key = (str(tensor.device), storage.data_ptr(), storage.nbytes())
        if key not in seen:
            seen.add(key)
            total += storage.nbytes()

    for parameter, state in optimizer.state.items():
        if isinstance(state.get("exp_avg"), OptimState8bit):
            quantized += parameter.numel()
        for value in state.values():
            if isinstance(value, OptimState8bit):
                add(value.codes); add(value.scale); add(value.qmap)
            elif torch.is_tensor(value):
                add(value)
    return total, quantized / PARAMETER_COUNT


def sample_parameter_vector(model: nn.Module, limit: int = 16_384) -> torch.Tensor:
    pieces, remaining = [], limit
    for parameter in model.parameters():
        if remaining <= 0:
            break
        flat = parameter.detach().reshape(-1)
        take = min(remaining, flat.numel())
        pieces.append(flat[:take].float())
        remaining -= take
    return torch.cat(pieces)


def log8_boundaries(optimizer) -> tuple[float | None, float | None, float | None]:
    states = [s for s in optimizer.state.values() if isinstance(s, dict)]
    values = [s["v"].reshape(-1) for s in states if "v" in s and torch.is_tensor(s["v"]) and s["v"].dtype == torch.int8]
    histories = [s["history"].reshape(-1) for s in states if "history" in s]
    if not values:
        return None, None, None
    v = torch.cat(values)
    ready = torch.cat(histories).ge(127).float().mean().item() if histories else None
    return v.eq(-128).float().mean().item(), v.eq(127).float().mean().item(), ready


@torch.inference_mode()
def evaluate(model: nn.Module, dataset: Dataset, batches: int | None) -> dict[str, float]:
    model.eval()
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
    total_loss, total_tokens = 0.0, 0
    for index, (inputs, labels, real_tokens) in enumerate(loader):
        if batches is not None and index >= batches:
            break
        inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(inputs)
            loss_sum = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100, reduction="sum")
        n = int(real_tokens.sum())
        total_loss += float(loss_sum)
        total_tokens += n
    loss = total_loss / total_tokens
    return {"loss": loss, "perplexity": math.exp(min(20.0, loss)), "bits_per_token": loss / math.log(2), "tokens": total_tokens}


class CSVWriter:
    def __init__(self, path: Path):
        self.path = path
        self.handle = None
        self.writer = None

    def write(self, row: dict[str, Any]) -> None:
        if self.handle is None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            exists = self.path.exists() and self.path.stat().st_size > 0
            self.handle = self.path.open("a", newline="", encoding="utf-8")
            self.writer = csv.DictWriter(self.handle, fieldnames=list(row))
            if not exists:
                self.writer.writeheader()
        self.writer.writerow(row)
        self.handle.flush()

    def close(self) -> None:
        if self.handle:
            self.handle.close()


## Shared training engine

Final runs are checkpointed at batch boundaries. Resume restores model,
optimizer, scaler, counters, and RNG state. The sampler then recreates the same
deterministic permutation and skips exactly the already-completed examples.


In [ ]:
def train_run(name: str, peak_lr: float, mode: str, max_tokens: int | None = None) -> dict[str, Any]:
    run_id = f"{mode}_{name}_lr{peak_lr:g}"
    out = RUN_ROOT / run_id
    out.mkdir(parents=True, exist_ok=True)
    checkpoint = out / "checkpoint.pt"
    completed = out / "summary.json"
    if completed.exists():
        return json.loads(completed.read_text())

    seed_everything(CFG.final_seed if mode == "final" else CFG.tune_seed)
    model = TinyStoriesLM().to(DEVICE)
    initial_state = FINAL_INITIAL_STATE if mode == "final" else TUNE_INITIAL_STATE
    initial_hash = FINAL_INITIAL_HASH if mode == "final" else TUNE_INITIAL_HASH
    model.load_state_dict(initial_state)
    optimizer = make_optimizer(name, model, peak_lr)
    scaler = torch.amp.GradScaler("cuda", enabled=not BF16)
    batches_per_epoch = math.ceil(len(TRAIN_DATA) / CFG.batch_size)
    full_steps = CFG.epochs * batches_per_epoch
    tune_steps = math.ceil((max_tokens or CFG.tune_tokens) / (CFG.batch_size * CFG.sequence_length))
    total_steps = full_steps if mode == "final" else min(batches_per_epoch, tune_steps)
    epoch, next_batch, global_step, tokens_seen = 0, 0, 0, 0
    best_val, best_step = float("inf"), -1

    if checkpoint.exists():
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        epoch, next_batch = state["epoch"], state["next_batch"]
        global_step, tokens_seen = state["global_step"], state["tokens_seen"]
        best_val, best_step = state["best_val"], state["best_step"]
        torch.set_rng_state(state["torch_rng"].cpu())
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda_rng"]])
        print("Resumed", run_id, "at", epoch, next_batch, global_step)

    # Remove telemetry beyond the last durable checkpoint so resumed CSV files
    # contain one row per event rather than duplicated post-checkpoint work.
    for csv_path in (out / "steps.csv", out / "evaluations.csv"):
        if csv_path.exists():
            if checkpoint.exists():
                frame = pd.read_csv(csv_path)
                frame = frame[frame["global_step"] <= global_step]
                frame.to_csv(csv_path, index=False)
            else:
                csv_path.unlink()

    step_log = CSVWriter(out / "steps.csv")
    eval_log = CSVWriter(out / "evaluations.csv")
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    ema_loss = None
    losses_tail = []

    while epoch < (CFG.epochs if mode == "final" else 1) and global_step < total_steps:
        start_example = next_batch * CFG.batch_size
        sampler = FixedEpochSampler(len(TRAIN_DATA), CFG.data_seed, epoch, start_example)
        loader = DataLoader(
            TRAIN_DATA, batch_size=CFG.batch_size, sampler=sampler,
            num_workers=CFG.num_workers, pin_memory=True, drop_last=False,
            persistent_workers=CFG.num_workers > 0,
        )
        for local_batch, (inputs, labels, real_tokens) in enumerate(loader, start=next_batch):
            if global_step >= total_steps:
                break
            model.train()
            lr = scheduled_lr(peak_lr, global_step, total_steps)
            set_optimizer_lr(optimizer, lr)
            inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            log_due = global_step == 0 or (global_step + 1) % CFG.log_every == 0
            sampled_before = sample_parameter_vector(model).clone() if log_due else None
            tick = time.perf_counter()
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(inputs)
                loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if name == COMPILED_OPTIMIZER:
                grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf")))
            else:
                grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm))
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.get_scale() < old_scale
            torch.cuda.synchronize()
            step_ms = (time.perf_counter() - tick) * 1000
            actual_tokens = int(real_tokens.sum())
            if not skipped:
                tokens_seen += actual_tokens
            global_step += 1
            next_batch = local_batch + 1
            value = float(loss.detach())
            ema_loss = value if ema_loss is None else 0.98 * ema_loss + 0.02 * value
            losses_tail.append(value)
            losses_tail = losses_tail[-200:]

            if global_step == 1 or global_step % CFG.log_every == 0:
                lower, upper, ready = log8_boundaries(optimizer)
                sampled_after = sample_parameter_vector(model)
                parameter_norm = float(sampled_after.norm())
                update_norm = float((sampled_after - sampled_before).norm())
                state_bytes, quantized_fraction = optimizer_state_audit(optimizer)
                step_log.write({
                    "mode": mode, "optimizer": name, "peak_lr": peak_lr,
                    "epoch": epoch, "batch_in_epoch": local_batch,
                    "global_step": global_step, "tokens_seen": tokens_seen,
                    "lr": lr, "train_loss": value, "train_loss_ema": ema_loss,
                    "grad_norm_preclip": grad_norm, "amp_skipped": skipped,
                    "sampled_parameter_norm": parameter_norm,
                    "sampled_update_norm": update_norm,
                    "sampled_update_to_parameter_ratio": update_norm / max(parameter_norm, 1e-30),
                    "step_ms": step_ms, "tokens_per_second": actual_tokens / (step_ms / 1000),
                    "gpu_allocated_bytes": torch.cuda.memory_allocated(),
                    "gpu_reserved_bytes": torch.cuda.memory_reserved(),
                    "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                    "cpu_rss_bytes": psutil.Process().memory_info().rss,
                    "state_tensor_bytes": state_bytes,
                    "state_bytes_per_parameter": state_bytes / PARAMETER_COUNT,
                    "quantized_parameter_fraction": quantized_fraction,
                    "log8_lower_fraction": lower, "log8_upper_fraction": upper,
                    "history_ready_fraction": ready,
                    "compile_failures": getattr(optimizer, "compile_failures", None),
                    "compilation_active": getattr(optimizer, "compilation_active", None),
                })

            should_eval = global_step == total_steps or global_step % CFG.eval_every == 0
            if should_eval:
                metrics = evaluate(model, VALID_DATA, CFG.eval_batches)
                if metrics["loss"] < best_val:
                    best_val, best_step = metrics["loss"], global_step
                    torch.save(model.state_dict(), out / "best_model.pt")
                eval_log.write({
                    "mode": mode, "optimizer": name, "peak_lr": peak_lr,
                    "epoch": epoch, "global_step": global_step, "tokens_seen": tokens_seen,
                    "validation_loss": metrics["loss"], "validation_perplexity": metrics["perplexity"],
                    "validation_bits_per_token": metrics["bits_per_token"],
                    "best_validation_loss": best_val, "best_step": best_step,
                })
                print(run_id, global_step, f"train={ema_loss:.4f}", f"val={metrics['loss']:.4f}")

            if mode == "final" and (global_step % CFG.checkpoint_every == 0):
                torch.save({
                    "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch, "next_batch": next_batch,
                    "global_step": global_step, "tokens_seen": tokens_seen,
                    "best_val": best_val, "best_step": best_step,
                    "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
                }, checkpoint)

        epoch += 1
        next_batch = 0

    final_val = evaluate(model, VALID_DATA, None if mode == "final" else CFG.eval_batches)
    state_bytes, quantized_fraction = optimizer_state_audit(optimizer)
    result = {
        "mode": mode, "optimizer": name, "peak_lr": peak_lr,
        "epochs_completed": epoch, "steps": global_step, "tokens_seen": tokens_seen,
        "best_validation_loss": best_val, "best_validation_step": best_step,
        "final_validation_loss": final_val["loss"],
        "final_validation_perplexity": final_val["perplexity"],
        "state_tensor_bytes": state_bytes,
        "state_bytes_per_parameter": state_bytes / PARAMETER_COUNT,
        "quantized_parameter_fraction": quantized_fraction,
        "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(),
        "wall_seconds": time.perf_counter() - started,
        "tail_train_loss_mean": float(np.mean(losses_tail)),
        "tail_train_loss_slope": float(np.polyfit(np.arange(len(losses_tail)), losses_tail, 1)[0]) if len(losses_tail) > 2 else None,
        "initial_state_sha256": initial_hash,
        "compiled_version": getattr(logrms_compiled, "COMPILED_VERSION", None),
        "compile_failures": getattr(optimizer, "compile_failures", None),
        "compilation_active": getattr(optimizer, "compilation_active", None),
        "dynamo_unique_graphs": int(torch._dynamo.utils.counters["stats"]["unique_graphs"]),
        "dynamo_graph_breaks": int(sum(torch._dynamo.utils.counters["graph_break"].values())),
    }
    if mode == "final":
        test = evaluate(model, TEST_DATA, None)
        result.update({f"test_{key}": value for key, value in test.items()})
        result["exact_protocol_epochs"] = CFG.epochs
        result["convergence_note"] = (
            "Training loss was still falling at the two-epoch boundary."
            if (result["tail_train_loss_slope"] or 0) < -1e-5
            else "Training loss was approximately flat at the two-epoch boundary."
        )
    atomic_json(completed, result)
    if checkpoint.exists():
        checkpoint.unlink()
    step_log.close(); eval_log.close()
    del model, optimizer, scaler
    gc.collect(); torch.cuda.empty_cache()
    return result


## Compiled final run and reference comparison

Recorded reference telemetry is embedded below. The cell trains only the compiled implementation, verifies the shared protocol, and compares telemetry at matching steps and token counts. No earlier Drive run is required.


In [ ]:
REFERENCE_SUMMARY = {
  "optimizer": "logrms_reference",
  "peak_lr": 0.003,
  "epochs_completed": 2,
  "steps": 115178,
  "tokens_seen": 943531918,
  "best_validation_loss": 1.60078,
  "best_validation_step": 115178,
  "final_validation_loss": 1.577972,
  "final_validation_perplexity": 4.845118,
  "wall_seconds": 12570.289891,
  "tail_train_loss_mean": 1.552613,
  "tail_train_loss_slope": -2.1e-05,
  "test_loss": 1.543446,
  "test_perplexity": 4.680694,
  "test_bits_per_token": 2.226722,
  "test_tokens": 2481710,
  "exact_protocol_epochs": 2,
  "state_bytes_per_parameter": 2.0,
  "initial_state_sha256_visible_prefix": "b48258196b044a240d436fee1fc308f79441afa99e06dc",
  "source": "User-supplied Colab output recorded 5M reference telemetry",
  "curve_decimal_precision": 4,
  "summary_decimal_precision": 6
}
REFERENCE_CURVE = [{"global_step": 1000, "train_loss_ema": 3.5963, "validation_loss": 3.5649}, {"global_step": 2000, "train_loss_ema": 2.7866, "validation_loss": 2.8008}, {"global_step": 3000, "train_loss_ema": 2.4772, "validation_loss": 2.496}, {"global_step": 4000, "train_loss_ema": 2.3113, "validation_loss": 2.3331}, {"global_step": 5000, "train_loss_ema": 2.2441, "validation_loss": 2.2654}, {"global_step": 6000, "train_loss_ema": 2.1726, "validation_loss": 2.2081}, {"global_step": 7000, "train_loss_ema": 2.0834, "validation_loss": 2.1271}, {"global_step": 8000, "train_loss_ema": 2.0474, "validation_loss": 2.0826}, {"global_step": 9000, "train_loss_ema": 2.0105, "validation_loss": 2.0568}, {"global_step": 10000, "train_loss_ema": 1.9926, "validation_loss": 2.0167}, {"global_step": 11000, "train_loss_ema": 1.9644, "validation_loss": 1.9927}, {"global_step": 12000, "train_loss_ema": 1.9334, "validation_loss": 1.9727}, {"global_step": 13000, "train_loss_ema": 1.9215, "validation_loss": 1.9692}, {"global_step": 14000, "train_loss_ema": 1.9162, "validation_loss": 1.9415}, {"global_step": 15000, "train_loss_ema": 1.8974, "validation_loss": 1.9262}, {"global_step": 16000, "train_loss_ema": 1.8798, "validation_loss": 1.9137}, {"global_step": 17000, "train_loss_ema": 1.869, "validation_loss": 1.9007}, {"global_step": 18000, "train_loss_ema": 1.8659, "validation_loss": 1.8908}, {"global_step": 19000, "train_loss_ema": 1.8398, "validation_loss": 1.8801}, {"global_step": 20000, "train_loss_ema": 1.8462, "validation_loss": 1.8735}, {"global_step": 21000, "train_loss_ema": 1.8304, "validation_loss": 1.8659}, {"global_step": 22000, "train_loss_ema": 1.8202, "validation_loss": 1.8566}, {"global_step": 23000, "train_loss_ema": 1.8184, "validation_loss": 1.8589}, {"global_step": 24000, "train_loss_ema": 1.8163, "validation_loss": 1.8452}, {"global_step": 25000, "train_loss_ema": 1.8146, "validation_loss": 1.8431}, {"global_step": 26000, "train_loss_ema": 1.8038, "validation_loss": 1.8353}, {"global_step": 27000, "train_loss_ema": 1.7933, "validation_loss": 1.832}, {"global_step": 28000, "train_loss_ema": 1.7887, "validation_loss": 1.8241}, {"global_step": 29000, "train_loss_ema": 1.7809, "validation_loss": 1.8256}, {"global_step": 30000, "train_loss_ema": 1.7748, "validation_loss": 1.8134}, {"global_step": 31000, "train_loss_ema": 1.7805, "validation_loss": 1.8158}, {"global_step": 32000, "train_loss_ema": 1.7773, "validation_loss": 1.8124}, {"global_step": 33000, "train_loss_ema": 1.7653, "validation_loss": 1.8067}, {"global_step": 34000, "train_loss_ema": 1.766, "validation_loss": 1.8014}, {"global_step": 35000, "train_loss_ema": 1.7531, "validation_loss": 1.7975}, {"global_step": 36000, "train_loss_ema": 1.7596, "validation_loss": 1.7919}, {"global_step": 37000, "train_loss_ema": 1.7494, "validation_loss": 1.7904}, {"global_step": 38000, "train_loss_ema": 1.7438, "validation_loss": 1.7867}, {"global_step": 39000, "train_loss_ema": 1.7357, "validation_loss": 1.7845}, {"global_step": 40000, "train_loss_ema": 1.7379, "validation_loss": 1.7766}, {"global_step": 41000, "train_loss_ema": 1.7318, "validation_loss": 1.7794}, {"global_step": 42000, "train_loss_ema": 1.723, "validation_loss": 1.7723}, {"global_step": 43000, "train_loss_ema": 1.7212, "validation_loss": 1.7668}, {"global_step": 44000, "train_loss_ema": 1.7227, "validation_loss": 1.7654}, {"global_step": 45000, "train_loss_ema": 1.7212, "validation_loss": 1.7628}, {"global_step": 46000, "train_loss_ema": 1.7155, "validation_loss": 1.7591}, {"global_step": 47000, "train_loss_ema": 1.7212, "validation_loss": 1.7569}, {"global_step": 48000, "train_loss_ema": 1.7126, "validation_loss": 1.7529}, {"global_step": 49000, "train_loss_ema": 1.7124, "validation_loss": 1.7475}, {"global_step": 50000, "train_loss_ema": 1.704, "validation_loss": 1.7466}, {"global_step": 51000, "train_loss_ema": 1.7046, "validation_loss": 1.7446}, {"global_step": 52000, "train_loss_ema": 1.6922, "validation_loss": 1.7383}, {"global_step": 53000, "train_loss_ema": 1.6949, "validation_loss": 1.7364}, {"global_step": 54000, "train_loss_ema": 1.6939, "validation_loss": 1.7344}, {"global_step": 55000, "train_loss_ema": 1.684, "validation_loss": 1.7319}, {"global_step": 56000, "train_loss_ema": 1.6954, "validation_loss": 1.7316}, {"global_step": 57000, "train_loss_ema": 1.6852, "validation_loss": 1.7274}, {"global_step": 58000, "train_loss_ema": 1.6684, "validation_loss": 1.7282}, {"global_step": 59000, "train_loss_ema": 1.6639, "validation_loss": 1.7233}, {"global_step": 60000, "train_loss_ema": 1.6721, "validation_loss": 1.7205}, {"global_step": 61000, "train_loss_ema": 1.6801, "validation_loss": 1.7181}, {"global_step": 62000, "train_loss_ema": 1.6533, "validation_loss": 1.7163}, {"global_step": 63000, "train_loss_ema": 1.6744, "validation_loss": 1.7154}, {"global_step": 64000, "train_loss_ema": 1.6663, "validation_loss": 1.7162}, {"global_step": 65000, "train_loss_ema": 1.6703, "validation_loss": 1.715}, {"global_step": 66000, "train_loss_ema": 1.6619, "validation_loss": 1.7087}, {"global_step": 67000, "train_loss_ema": 1.6611, "validation_loss": 1.7051}, {"global_step": 68000, "train_loss_ema": 1.661, "validation_loss": 1.6997}, {"global_step": 69000, "train_loss_ema": 1.6651, "validation_loss": 1.6981}, {"global_step": 70000, "train_loss_ema": 1.6442, "validation_loss": 1.6922}, {"global_step": 71000, "train_loss_ema": 1.6379, "validation_loss": 1.6906}, {"global_step": 72000, "train_loss_ema": 1.6542, "validation_loss": 1.6869}, {"global_step": 73000, "train_loss_ema": 1.6505, "validation_loss": 1.6834}, {"global_step": 74000, "train_loss_ema": 1.6286, "validation_loss": 1.6794}, {"global_step": 75000, "train_loss_ema": 1.6255, "validation_loss": 1.6766}, {"global_step": 76000, "train_loss_ema": 1.6351, "validation_loss": 1.6739}, {"global_step": 77000, "train_loss_ema": 1.6366, "validation_loss": 1.6716}, {"global_step": 78000, "train_loss_ema": 1.6302, "validation_loss": 1.6685}, {"global_step": 79000, "train_loss_ema": 1.6138, "validation_loss": 1.6674}, {"global_step": 80000, "train_loss_ema": 1.6165, "validation_loss": 1.6649}, {"global_step": 81000, "train_loss_ema": 1.6216, "validation_loss": 1.6596}, {"global_step": 82000, "train_loss_ema": 1.6156, "validation_loss": 1.6564}, {"global_step": 83000, "train_loss_ema": 1.6034, "validation_loss": 1.6534}, {"global_step": 84000, "train_loss_ema": 1.6014, "validation_loss": 1.6499}, {"global_step": 85000, "train_loss_ema": 1.6119, "validation_loss": 1.6478}, {"global_step": 86000, "train_loss_ema": 1.603, "validation_loss": 1.6455}, {"global_step": 87000, "train_loss_ema": 1.5976, "validation_loss": 1.6419}, {"global_step": 88000, "train_loss_ema": 1.5991, "validation_loss": 1.6404}, {"global_step": 89000, "train_loss_ema": 1.5995, "validation_loss": 1.6365}, {"global_step": 90000, "train_loss_ema": 1.5866, "validation_loss": 1.6344}, {"global_step": 91000, "train_loss_ema": 1.5916, "validation_loss": 1.6317}, {"global_step": 92000, "train_loss_ema": 1.5893, "validation_loss": 1.6302}, {"global_step": 93000, "train_loss_ema": 1.5785, "validation_loss": 1.6268}, {"global_step": 94000, "train_loss_ema": 1.5777, "validation_loss": 1.6244}, {"global_step": 95000, "train_loss_ema": 1.5806, "validation_loss": 1.6234}, {"global_step": 96000, "train_loss_ema": 1.5835, "validation_loss": 1.6208}, {"global_step": 97000, "train_loss_ema": 1.5739, "validation_loss": 1.6189}, {"global_step": 98000, "train_loss_ema": 1.5587, "validation_loss": 1.6169}, {"global_step": 99000, "train_loss_ema": 1.5645, "validation_loss": 1.615}, {"global_step": 100000, "train_loss_ema": 1.5638, "validation_loss": 1.6136}, {"global_step": 101000, "train_loss_ema": 1.5698, "validation_loss": 1.6126}, {"global_step": 102000, "train_loss_ema": 1.5638, "validation_loss": 1.6108}, {"global_step": 103000, "train_loss_ema": 1.5602, "validation_loss": 1.6093}, {"global_step": 104000, "train_loss_ema": 1.5586, "validation_loss": 1.608}, {"global_step": 105000, "train_loss_ema": 1.5704, "validation_loss": 1.6068}, {"global_step": 106000, "train_loss_ema": 1.5503, "validation_loss": 1.6057}, {"global_step": 107000, "train_loss_ema": 1.5596, "validation_loss": 1.6048}, {"global_step": 108000, "train_loss_ema": 1.5705, "validation_loss": 1.6038}, {"global_step": 109000, "train_loss_ema": 1.5693, "validation_loss": 1.6032}, {"global_step": 110000, "train_loss_ema": 1.5499, "validation_loss": 1.6027}, {"global_step": 111000, "train_loss_ema": 1.5615, "validation_loss": 1.602}, {"global_step": 112000, "train_loss_ema": 1.5488, "validation_loss": 1.6017}, {"global_step": 113000, "train_loss_ema": 1.5583, "validation_loss": 1.6014}, {"global_step": 114000, "train_loss_ema": 1.55, "validation_loss": 1.6009}, {"global_step": 115000, "train_loss_ema": 1.5663, "validation_loss": 1.6008}, {"global_step": 115178, "train_loss_ema": 1.5564, "validation_loss": 1.6008}]
reference_curve = pd.DataFrame(REFERENCE_CURVE)

if not FINAL_INITIAL_HASH.startswith(
    REFERENCE_SUMMARY["initial_state_sha256_visible_prefix"]
):
    raise RuntimeError(
        "The reconstructed initial weights do not match the visible SHA-256 prefix from the "
        "reference run. This usually indicates a changed model or PyTorch initialization."
    )

compiled_summary = train_run(COMPILED_OPTIMIZER, PEAK_LR, "final")
expected_protocol = {
    "steps": REFERENCE_SUMMARY["steps"],
    "tokens_seen": REFERENCE_SUMMARY["tokens_seen"],
    "exact_protocol_epochs": REFERENCE_SUMMARY["exact_protocol_epochs"],
}
for field, expected in expected_protocol.items():
    if compiled_summary[field] != expected:
        raise RuntimeError(
            f"Protocol mismatch for {field}: expected={expected!r}, "
            f"compiled={compiled_summary[field]!r}"
        )
if compiled_summary["initial_state_sha256"] != FINAL_INITIAL_HASH:
    raise RuntimeError("Compiled run did not use the registered initial weights")
if compiled_summary["compile_failures"] != 0:
    raise RuntimeError("The compiled run recorded a compilation failure")
if not compiled_summary["compilation_active"]:
    raise RuntimeError("Compilation was not active at the end of training")
if not math.isclose(float(compiled_summary["peak_lr"]), PEAK_LR, rel_tol=0, abs_tol=0):
    raise RuntimeError("Compiled run used the wrong learning rate")

COMPILED_RUN = RUN_ROOT / f"final_{COMPILED_OPTIMIZER}_lr{PEAK_LR:g}"
compiled_steps = pd.read_csv(COMPILED_RUN / "steps.csv")
compiled_eval = pd.read_csv(COMPILED_RUN / "evaluations.csv")

step_comparison = reference_curve[["global_step", "train_loss_ema"]].merge(
    compiled_steps[["global_step", "tokens_seen", "train_loss_ema", "step_ms"]],
    on="global_step",
    suffixes=("_reference", "_compiled"),
    validate="one_to_one",
)
step_comparison["train_loss_ema_delta"] = (
    step_comparison["train_loss_ema_compiled"] - step_comparison["train_loss_ema_reference"]
)
step_comparison.to_csv(RUN_ROOT / "compiled_vs_reference_steps.csv", index=False)

evaluation_comparison = reference_curve[["global_step", "validation_loss"]].merge(
    compiled_eval[["global_step", "tokens_seen", "validation_loss"]],
    on="global_step",
    suffixes=("_reference", "_compiled"),
    validate="one_to_one",
)
evaluation_comparison["validation_loss_delta"] = (
    evaluation_comparison["validation_loss_compiled"]
    - evaluation_comparison["validation_loss_reference"]
)
evaluation_comparison.to_csv(
    RUN_ROOT / "compiled_vs_reference_evaluations.csv", index=False
)

comparison_metrics = {
    "recorded_curve_points": len(reference_curve),
    "matched_training_points": len(step_comparison),
    "matched_evaluation_points": len(evaluation_comparison),
    "mean_abs_train_loss_ema_delta": float(step_comparison.train_loss_ema_delta.abs().mean()),
    "max_abs_train_loss_ema_delta": float(step_comparison.train_loss_ema_delta.abs().max()),
    "mean_abs_validation_loss_delta": float(evaluation_comparison.validation_loss_delta.abs().mean()),
    "max_abs_validation_loss_delta": float(evaluation_comparison.validation_loss_delta.abs().max()),
    "final_validation_loss_delta": float(
        compiled_summary["final_validation_loss"]
        - REFERENCE_SUMMARY["final_validation_loss"]
    ),
    "test_loss_delta": float(
        compiled_summary["test_loss"] - REFERENCE_SUMMARY["test_loss"]
    ),
    "wall_time_speedup": float(
        REFERENCE_SUMMARY["wall_seconds"] / compiled_summary["wall_seconds"]
    ),
    "compiled_state_bytes_per_parameter": compiled_summary["state_bytes_per_parameter"],
    "dynamo_unique_graphs": compiled_summary["dynamo_unique_graphs"],
    "dynamo_graph_breaks": compiled_summary["dynamo_graph_breaks"],
    "recorded_curve_decimal_precision": 4,
}
display(pd.DataFrame([
    {"implementation": "reference_eager_recorded", **REFERENCE_SUMMARY},
    {"implementation": "torch_compile", **compiled_summary},
]))
display(pd.DataFrame([comparison_metrics]))


## Trajectory overlay and export


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(reference_curve.global_step, reference_curve.train_loss_ema, label="eager reference")
axes[0].plot(compiled_steps.global_step, compiled_steps.train_loss_ema, label="torch.compile", alpha=0.85)
axes[1].plot(reference_curve.global_step, reference_curve.validation_loss, marker="o", label="eager reference")
axes[1].plot(compiled_eval.global_step, compiled_eval.validation_loss, marker="o", label="torch.compile", alpha=0.85)
axes[0].set(title="Training-loss trajectory", xlabel="global step", ylabel="cross entropy")
axes[1].set(title="Validation-loss trajectory", xlabel="global step", ylabel="cross entropy")
for axis in axes:
    axis.grid(alpha=0.25); axis.legend()
fig.tight_layout()
plot_path = RUN_ROOT / "compiled_vs_reference_trajectories.png"
fig.savefig(plot_path, dpi=160)
plt.show()

comparison_manifest = {
    "notebook_build": NOTEBOOK_BUILD,
    "reference_source": REFERENCE_SUMMARY["source"],
    "recorded_curve_decimal_precision": 4,
    "reference_source_sha256": LOGRMS_REFERENCE_SHA256,
    "compiled_source_sha256": sha256_file(COMPILED_PATH),
    "initial_state_sha256": FINAL_INITIAL_HASH,
    "peak_lr": PEAK_LR,
    "epochs": CFG.epochs,
    "steps": compiled_summary["steps"],
    "tokens_seen": compiled_summary["tokens_seen"],
    "comparison_metrics": comparison_metrics,
}
atomic_json(RUN_ROOT / "compiled_vs_reference_manifest.json", comparison_manifest)

export = DRIVE_ROOT / "compiled_embedded_reference_export_20260712_A"
export.mkdir(parents=True, exist_ok=True)
(export / "reference_summary.json").write_text(
    json.dumps(REFERENCE_SUMMARY, indent=2), encoding="utf-8"
)
reference_curve.to_csv(export / "reference_curve.csv", index=False)
for path in [
    COMPILED_RUN / "summary.json", COMPILED_RUN / "steps.csv", COMPILED_RUN / "evaluations.csv",
    RUN_ROOT / "compiled_vs_reference_steps.csv",
    RUN_ROOT / "compiled_vs_reference_evaluations.csv",
    RUN_ROOT / "compiled_vs_reference_manifest.json", plot_path,
]:
    shutil.copy2(path, export / path.name)
archive = shutil.make_archive(
    str(DRIVE_ROOT / "logrms_5m_compiled_parity_results"),
    "zip", export,
)
print("Compiled comparison archive:", archive)
